In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

In [2]:
master = pd.read_csv('C:/Users/shere/OneDrive/Desktop/ecommerce-sales-intelligence/data/processed/master.csv')
master['order_purchase_timestamp'] = pd.to_datetime(master['order_purchase_timestamp'])


In [3]:
print(f"Rows: {len(master):,}")
print("Ready!")

Rows: 115,723
Ready!


In [4]:
total_revenue = master['payment_value'].sum()
total_orders  = master['order_id'].nunique()
avg_review    = master['review_score'].mean()
avg_delivery  = master['delivery_days'].mean()

In [5]:
print(f"Total Revenue    : R$ {total_revenue:,.0f}")
print(f"Total Orders     : {total_orders:,}")
print(f"Avg Review Score : {avg_review:.2f} / 5.0")
print(f"Avg Delivery Days: {avg_delivery:.1f} days")

Total Revenue    : R$ 19,881,945
Total Orders     : 96,478
Avg Review Score : 4.08 / 5.0
Avg Delivery Days: 12.0 days


In [6]:
# Save for dashboard
kpis = pd.DataFrame([{
    'total_revenue': round(total_revenue, 2),
    'total_orders': total_orders,
    'avg_review': round(avg_review, 2),
    'avg_delivery_days': round(avg_delivery, 1)
}])
kpis.to_csv('../data/processed/kpis.csv', index=False)
print("Saved kpis.csv")

Saved kpis.csv


In [7]:
delivery_vs_review = master.groupby('review_score')['delivery_days'].mean().reset_index()
delivery_vs_review.columns = ['review_score', 'avg_delivery_days']

fig = px.bar(delivery_vs_review, x='review_score', y='avg_delivery_days',
    title='Avg Delivery Days per Review Score',
    color='avg_delivery_days', color_continuous_scale='RdYlGn_r')
fig.show()

delivery_vs_review.to_csv('../data/processed/review_vs_delivery.csv', index=False)
print("Saved review_vs_delivery.csv")

Saved review_vs_delivery.csv


In [8]:
master['day_of_week'] = master['order_purchase_timestamp'].dt.day_name()
master['hour'] = master['order_purchase_timestamp'].dt.hour

day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
orders_by_day = master.groupby('day_of_week')['order_id'].nunique().reindex(day_order).reset_index()
orders_by_day.columns = ['day', 'orders']

fig = px.bar(orders_by_day, x='day', y='orders',
    title='Orders by Day of Week',
    color='orders', color_continuous_scale='Blues')
fig.show()

orders_by_day.to_csv('../data/processed/orders_by_day.csv', index=False)
print("Saved orders_by_day.csv")

Saved orders_by_day.csv


In [9]:
orders_by_hour = master.groupby('hour')['order_id'].nunique().reset_index()
orders_by_hour.columns = ['hour', 'orders']

fig = px.line(orders_by_hour, x='hour', y='orders',
    title='Orders by Hour of Day', markers=True)
fig.show()

orders_by_hour.to_csv('../data/processed/orders_by_hour.csv', index=False)
print("Saved orders_by_hour.csv")

Saved orders_by_hour.csv


In [10]:
top_sellers = master.groupby('seller_id').agg(
    total_revenue=('payment_value', 'sum'),
    total_orders=('order_id', 'nunique'),
    avg_review=('review_score', 'mean'),
    avg_delivery=('delivery_days', 'mean')
).reset_index().sort_values('total_revenue', ascending=False).head(20)

top_sellers = top_sellers.round(2)
top_sellers.to_csv('../data/processed/top_sellers.csv', index=False)
print("Saved top_sellers.csv")
print(top_sellers[['seller_id','total_revenue','total_orders','avg_review']].to_string(index=False))

Saved top_sellers.csv
                       seller_id  total_revenue  total_orders  avg_review
7c67e1448b00f6e969d365cea6b010ab      510915.44           973        3.40
1025f0e2d44d7041d6cf58b6550e0bfa      310186.70           910        3.88
4a3ca9315b744ce9f8e9374361493884      300724.29          1772        3.82
1f50f920176fa81dab994f9023523100      291526.94          1399        3.99
53243585a1d6dc2643021fd1853d8905      279843.42           348        4.12
da8622b14eb17ae2831f4ac5b9dab84a      276093.09          1311        4.08
4869f7a5dfa277a7dca6462dcf3b52b2      261532.48          1124        4.12
955fee9216a65b617aa5c0531780ce60      232228.24          1261        4.09
fa1c13f2614d7b5c4749cbc52fecda94      203262.00           578        4.38
6560211a19b47992c3666cc44a7e94c0      176467.46          1819        3.95
7e93a43ef30c4f03f38b393420bc753a      174053.79           319        4.36
7a67c85e85bb2ce8582c35f2203ad736      167205.00          1145        4.26
25c5c91f63607446

In [11]:
monthly  = pd.read_csv('../data/processed/monthly_revenue.csv')
by_state = pd.read_csv('../data/processed/revenue_by_state.csv')
del_rev  = pd.read_csv('../data/processed/review_vs_delivery.csv')

top_month     = monthly.loc[monthly['revenue'].idxmax(), 'year_month']
top_month_rev = monthly['revenue'].max()
top_state     = by_state.iloc[0]['state']
top_state_pct = by_state.iloc[0]['total_orders'] / by_state['total_orders'].sum() * 100
one_star_days  = del_rev.loc[del_rev['review_score']==1, 'avg_delivery_days'].values[0]
five_star_days = del_rev.loc[del_rev['review_score']==5, 'avg_delivery_days'].values[0]

print(f"""
1. REVENUE PEAK
   Best month: {top_month} — R${top_month_rev:,.0f}
   November spike = Black Friday effect

2. GEOGRAPHIC CONCENTRATION
   {top_state} = {top_state_pct:.0f}% of all orders

3. DELIVERY DRIVES SATISFACTION
   1-star reviews: avg {one_star_days:.0f} days to deliver
   5-star reviews: avg {five_star_days:.0f} days to deliver

4. PAYMENT BEHAVIOUR
   Credit card dominates ~75% of orders

5. CATEGORY INSIGHT
   High-revenue categories ≠ high review scores
""")


1. REVENUE PEAK
   Best month: 2017-11 — R$1,559,740
   November spike = Black Friday effect

2. GEOGRAPHIC CONCENTRATION
   SP = 46% of all orders

3. DELIVERY DRIVES SATISFACTION
   1-star reviews: avg 19 days to deliver
   5-star reviews: avg 10 days to deliver

4. PAYMENT BEHAVIOUR
   Credit card dominates ~75% of orders

5. CATEGORY INSIGHT
   High-revenue categories ≠ high review scores

